# Eyewear brand localization / OCR

This notebook runs the standalone `eyewear-localization/` pipeline. It first tests OCR independently, then optionally enables class-agnostic SAM3 eyewear localization. Brand names are never sent to SAM3.

## 1. Clone and install

The notebook installs both OCR and native SAM3 dependencies in a UV environment. It then downloads the gated SAM3 checkpoint directly over HTTP using an approved Hugging Face token from a Kaggle Secret named `HF_TOKEN` (no `hf auth login` required).

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/fez-Ox/pxModel-Object-Counting.git"
REPO_DIR = Path("/kaggle/working/pxModel-localization")
APP_DIR = REPO_DIR / "eyewear-localization"
SAM3_APP = REPO_DIR / "sam3-verbose-counting"
SAM3_CHECKPOINT = SAM3_APP / "checkpoints" / "sam3.pt"

if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

# Install EasyOCR plus every dependency needed by the native SAM3 adapter.
subprocess.run(["uv", "sync", "--extra", "ocr", "--extra", "sam3"], cwd=APP_DIR, check=True)

def get_hf_token():
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None

# The downloader is direct HTTP and never writes the token to disk or runs `hf login`.
download_env = os.environ.copy()
hf_token = get_hf_token()
if hf_token:
    download_env["HF_TOKEN"] = hf_token
if not SAM3_CHECKPOINT.exists():
    download_command = [
        sys.executable, str(SAM3_APP / "download_model.py"),
        "--output", str(SAM3_CHECKPOINT),
        "--timeout", "600",
    ]
    subprocess.run(download_command, cwd=SAM3_APP, env=download_env, check=True)
else:
    print("Using existing checkpoint:", SAM3_CHECKPOINT)
print("Pipeline:", APP_DIR)
print("SAM3 checkpoint:", SAM3_CHECKPOINT)

## 2. Select an image

Edit `IMAGE_PATH` if the automatic first-image selection is not the image you want.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
IMAGE_PATH = Path("/kaggle/input/your-dataset/image.jpg")  # edit me
BRAND_FILE = APP_DIR / "brands.txt"  # edit or replace with your catalog
OUTPUT_DIR = APP_DIR / "outputs" / "notebook"
try:
    probe = subprocess.run(
        ["uv", "run", "python", "-c",
         "import torch; print('cuda' if torch.cuda.is_available() else 'cpu')"],
        cwd=APP_DIR, capture_output=True, text=True, check=True,
    )
    DEVICE = probe.stdout.strip()
except Exception:
    DEVICE = "cpu"

print("Image:", IMAGE_PATH)
print("Device:", DEVICE)
assert IMAGE_PATH.exists(), f"Update IMAGE_PATH: {IMAGE_PATH}"

In [ ]:
# ─── Tune pipeline parameters ────────────────────────────────────────
# Set any of these to override config.yaml values.  None = keep the
# config default.

TUNE = {
    # --- Fusion gate (acceptance thresholds) ---
    "tau": None,              # min probability to accept a brand (config.yaml: 0.50)
    "margin": None,           # min gap between best and runner-up (config.yaml: 0.15)
    "temperature": None,      # softmax temperature (config.yaml: 0.6)
    "unknown_prior": None,    # prior score for unknown (config.yaml: 0.40)
    "max_per_evidence": None, # cap on any single evidence contribution (config.yaml: 0.5)

    # --- Smoothing ---
    "smooth_lambda": None,    # pull toward neighbor brands (config.yaml: 0.25)
    "smooth_gate": None,      # only smooth from neighbors with p(unknown) below this (0.5)

    # --- Cue reliabilities (partial updates; unspecified cues keep config) ---
    "cue_reliability": {
        # "C1": 0.70,   # on-product OCR
        # "C2": 0.55,   # signage scope (lowered from 0.7 to reduce hallucination)
    },

    # --- C1 (on-product OCR) ---
    "c1_margin": None,        # crop margin fraction (default 0.25)
    "c1_scales": None,        # multi-scale factors, e.g. "2.0,4.0" (default: 2.0,4.0)
    "c1_no_sharpen": False,   # set True to disable post-upscale sharpening

    # --- OCR ---
    "ocr_scale": 2.0,        # upscale factor before EasyOCR (whole-image)

    # --- Scene filter ---
    "person_threshold": None, # SAM3 score for person detection (default 0.25)
    "poster_threshold": None, # SAM3 score for poster/ad detection (default 0.25)
    "shelf_threshold": None,  # SAM3 score for shelf detection (default 0.20)
    "shelf_filter": True,     # False to disable scene filtering entirely

    # --- Free-form overrides (dotted key=value strings) ---
    "set": [
        # "fusion.tau=0.5",
        # "cue_reliability.C1=0.8",
    ],
}


def _tune_cli_args():
    """Convert TUNE dict into CLI flags for infer.py."""
    args = []
    _map = {
        "tau": "--tau", "margin": "--margin",
        "temperature": "--temperature", "unknown_prior": "--unknown-prior",
        "max_per_evidence": "--max-per-evidence",
        "smooth_lambda": "--smooth-lambda", "smooth_gate": "--smooth-gate",
        "person_threshold": "--person-threshold",
        "poster_threshold": "--poster-threshold",
        "shelf_threshold": "--shelf-threshold",
    }
    for key, flag in _map.items():
        val = TUNE.get(key)
        if val is not None:
            args += [flag, str(val)]
    for cue, val in TUNE.get("cue_reliability", {}).items():
        args += ["--cue-reliability", f"{cue}={val}"]
    ocr_scale = TUNE.get("ocr_scale")
    if ocr_scale is not None:
        args += ["--ocr-scale", str(ocr_scale)]
    # C1 tunables
    c1_margin = TUNE.get("c1_margin")
    if c1_margin is not None:
        args += ["--c1-margin", str(c1_margin)]
    c1_scales = TUNE.get("c1_scales")
    if c1_scales is not None:
        args += ["--c1-scales", str(c1_scales)]
    if TUNE.get("c1_no_sharpen"):
        args.append("--c1-no-sharpen")
    for item in TUNE.get("set", []):
        args += ["--set", str(item)]
    if TUNE.get("shelf_filter") is False:
        args.append("--no-shelf-filter")
    return args

## 3. OCR-only test

This stage does not need SAM3. It detects all text, matches the configured brand gazetteer, and writes `text_detections[]` and `signs[]`. A sign is not assigned to an eyewear instance at this stage.

In [ ]:
def run_pipeline(checkpoint=None):
    command = [
        "uv", "run", "python", "infer.py", str(IMAGE_PATH),
        "--brand-file", str(BRAND_FILE),
        "--ocr-backend", "easyocr",
        "--device", DEVICE,
        "--no-vlm-audit",
        "--out", str(OUTPUT_DIR),
    ]
    command += _tune_cli_args()
    if checkpoint is not None:
        command += ["--sam3-checkpoint", str(checkpoint)]
    completed = subprocess.run(command, cwd=APP_DIR, text=True, capture_output=True)
    print(completed.stdout)
    if completed.returncode:
        print(completed.stderr)
        completed.check_returncode()
    result_path = OUTPUT_DIR / f"{IMAGE_PATH.stem}.json"
    result = json.loads(result_path.read_text())
    config_used = result.get("effective_config", {})
    print("Effective config:")
    for key in ("fusion", "smoothing", "cue_reliability"):
        print(f"  {key}: {config_used.get(key, {})}")
    return result


ocr_result = run_pipeline()
print("\nOCR text detections:")
for detection in ocr_result["text_detections"]:
    print(f"  {detection['text']!r} ({detection['confidence']:.2f})")
print("Gazetteer-matched signs:")
for sign in ocr_result["signs"]:
    print(f"  {sign['text']!r} -> {sign['brand']}")

In [ ]:
for sign in ocr_result["signs"]:
    scope = sign.get("scope", {})
    region = scope.get("region_bbox", [])
    print(f"  {sign['brand']} ({sign['text']}) -> {scope.get('type', '?')}")
    print(f"    region={region}  scope_conf={scope.get('confidence', 0):.2f}")

## 4. Full attribution test with SAM3

The setup cell has already downloaded the official checkpoint. It uses only class-agnostic eyewear prompts; brand names remain in the OCR/fusion stages.

In [ ]:
assert SAM3_CHECKPOINT.exists(), f"Checkpoint download failed: {SAM3_CHECKPOINT}"
full_result = run_pipeline(SAM3_CHECKPOINT)
print("\nPer-instance decisions (with gate diagnostics):")
for output in full_result["outputs"]:
    debug = output.get("decision_debug", {})
    gates = debug.get("gates", {})
    tau_ok = "OK" if gates.get("tau") else "FAIL"
    margin_ok = "OK" if gates.get("margin") else "FAIL"
    print(
        f"  {output['instance_id']} -> {output['brand']}"
        f"  probs={output['probabilities']}"
        f"  [tau={tau_ok} margin={margin_ok}]"
    )
print(f"\nExcluded by scene filter: {len(full_result['excluded_instances'])}")
for excluded in full_result["excluded_instances"]:
    print("  ", excluded["instance_id"], excluded["reasons"])

## 5. C2 column-boundary diagnostics

Inspect the inferred column boundaries and which instances were assigned to which brand column. This helps debug C2 hallucination — if an instance near a Moncler/Cartier boundary gets the wrong brand, the column boundary midpoint needs adjustment.

In [ ]:
print("=" * 72)
print("C2 COLUMN DIAGNOSTICS")
print("=" * 72)

# Extract column boundaries from evidence support.
columns_seen = {}  # (left, right) -> brand
for ev in full_result.get("evidence", []):
    if ev["cue"] != "C2":
        continue
    support = ev.get("support", {})
    col_left = support.get("column_left")
    col_right = support.get("column_right")
    if col_left is not None and col_right is not None:
        key = (round(col_left, 1), round(col_right, 1))
        columns_seen[key] = ev["brand"]

if columns_seen:
    print("\nInferred columns (from C2 evidence):")
    for (left, right), brand in sorted(columns_seen.items()):
        print(f"  [{left:.0f} – {right:.0f}] -> {brand}")
else:
    print("\n(No column info in C2 evidence — C2 may not have found header signs)")

# Per-instance C2 assignment.
print("\nPer-instance C2 assignment:")
c2_by_instance = {}
for ev in full_result.get("evidence", []):
    if ev["cue"] == "C2":
        c2_by_instance.setdefault(ev["instance_id"], []).append(ev)

for inst in full_result.get("instances", []):
    cx = inst["centroid"][0] if inst.get("centroid") else inst["bbox"][0] + inst["bbox"][2] / 2
    evs = c2_by_instance.get(inst["id"], [])
    if evs:
        brands = ", ".join(f"{e['brand']}({e['confidence']:.2f})" for e in evs)
        print(f"  {inst['id']} cx={cx:.0f} -> {brands}")
    else:
        print(f"  {inst['id']} cx={cx:.0f} -> (no C2 evidence)")

# C1 evidence summary.
c1_evs = [e for e in full_result.get("evidence", []) if e["cue"] == "C1"]
if c1_evs:
    print("\nC1 on-product detections:")
    for e in c1_evs:
        s = e.get("support", {})
        print(f"  {e['instance_id']} -> {e['brand']} conf={e['confidence']:.2f} "
              f"text={s.get('raw_text', '?')!r} scale={s.get('scale_used', '?')}")
else:
    print("\n(No C1 evidence — no on-product text detected)")

## 6. Visualize column boundaries

Draw the inferred column boundaries on the image to visually verify they correctly separate brand zones.

In [ ]:
from PIL import Image, ImageDraw, ImageFont

img = Image.open(IMAGE_PATH).convert("RGB")
draw = ImageDraw.Draw(img, "RGBA")
try:
    font = ImageFont.truetype("DejaVuSans.ttf", size=max(18, img.width // 50))
except Exception:
    font = ImageFont.load_default()

# Draw column boundaries as vertical dashed lines.
COLORS = [
    (255, 100, 100, 200),  # red
    (100, 255, 100, 200),  # green
    (100, 100, 255, 200),  # blue
    (255, 200, 50, 200),   # yellow
    (200, 100, 255, 200),  # purple
    (100, 255, 255, 200),  # cyan
]

if columns_seen:
    boundaries = set()
    for (left, right), brand in sorted(columns_seen.items()):
        boundaries.add(left)
        boundaries.add(right)
    for bx in sorted(boundaries):
        if 5 < bx < img.width - 5:  # skip image edges
            for y in range(0, img.height, 20):
                draw.line([(bx, y), (bx, min(y + 10, img.height))],
                          fill=(255, 255, 0, 180), width=3)

    # Label each column.
    for i, ((left, right), brand) in enumerate(sorted(columns_seen.items())):
        color = COLORS[i % len(COLORS)]
        mid_x = (left + right) / 2
        draw.text((mid_x - 30, 10), brand.upper(), fill=color, font=font)

# Draw instance boxes colored by their final brand.
outputs_by_id = {o["instance_id"]: o for o in full_result.get("outputs", [])}
for inst in full_result.get("instances", []):
    x, y, w, h = inst["bbox"]
    decision = outputs_by_id.get(inst["id"], {})
    brand = decision.get("brand", "unknown")
    color = (35, 180, 75, 230) if brand != "unknown" else (200, 100, 30, 200)
    draw.rectangle((x, y, x + w, y + h), outline=color, width=3)
    label = f"{inst['id']}: {brand}"
    text_box = draw.textbbox((x, y), label, font=font)
    draw.rectangle(text_box, fill=color)
    draw.text((x, y), label, fill=(255, 255, 255), font=font)

img.save(OUTPUT_DIR / f"{IMAGE_PATH.stem}_columns.jpg")
display(img.resize((img.width // 2, img.height // 2)))

## 7. Inspect the JSON contract

The result keeps raw OCR, scoped signs, cue evidence, and final decisions separate for auditing.

In [ ]:
from pprint import pprint
pprint({
    "signs": full_result["signs"],
    "evidence": full_result["evidence"],
    "outputs": full_result["outputs"],
})